In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 8.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 81.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=ebda05dba9b216b21a26e5512b92264997eccd6cfaf691eb46ca63ed725f994b
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [15]:
import sys
from importlib.metadata import version, requires
from packaging.requirements import Requirement
from packaging.markers import default_environment

import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_algorithms
import qiskit_machine_learning
import qiskit_optimization
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

expected = {
    'qiskit': '2.2.3',
    'qiskit-machine-learning': '0.9.0',
    'qiskit-optimization': '0.7.0',
    'qiskit-algorithms': '0.4.0',
}
print('Python:', sys.version.split()[0])
environment = default_environment()
environment['extra'] = ''
for package, target in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == target, f'{package} 설치 셀을 다시 실행하세요.'
    for specification in requires(package) or []:
        requirement = Requirement(specification)
        if requirement.marker and not requirement.marker.evaluate(environment):
            continue
        actual = version(requirement.name)
        assert actual in requirement.specifier, f'의존성 확인: {specification}, 설치 {actual}'
for package in ('numpy', 'scipy', 'scikit-learn'):
    print(package, version(package))
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
probability = Statevector.from_instruction(qc).probabilities()
assert np.allclose(probability, [0.5, 0, 0, 0.5])
print('회로 확률:', probability)
print('패키지 의존성·import·회로 연산 확인 완료')

Python: 3.13.18
qiskit 2.2.3
qiskit-machine-learning 0.9.0
qiskit-optimization 0.7.0
qiskit-algorithms 0.4.0
numpy 2.4.6
scipy 1.16.3
scikit-learn 1.6.1
회로 확률: [0.5 0.  0.  0.5]
패키지 의존성·import·회로 연산 확인 완료


In [16]:
import numpy as np

# 두 항목의 선택 보상 3점, 2점과 비용 2, 1을 사용합니다.
cost = np.array([2, 1])
reward = np.array([3.0, 2.0])
budget, penalty = 2, 6.0
for x, slack in [(np.array([1, 0]), 0),
                 (np.array([0, 1]), 1),
                 (np.array([1, 1]), 0)]:
    original = -float(reward @ x)
    residual = int(cost @ x) + slack - budget
    qubo = original + penalty * residual**2
    print('선택:', x.tolist(), '슬랙:', slack,
          '원래 목적값:', original, '잔차:', residual, 'QUBO:', qubo)

선택: [1, 0] 슬랙: 0 원래 목적값: -3.0 잔차: 0 QUBO: -3.0
선택: [0, 1] 슬랙: 1 원래 목적값: -2.0 잔차: 0 QUBO: -2.0
선택: [1, 1] 슬랙: 0 원래 목적값: -5.0 잔차: 1 QUBO: 1.0


In [18]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

gamma = np.pi / 2  # 감마: 비용 위상 각도, 라디안
beta = -np.pi / 4  # 베타: 믹서 각도, 라디안
qc = QuantumCircuit(1)
qc.h(0)
initial = Statevector.from_instruction(qc)
qc.p(-gamma, 0)
after_cost = Statevector.from_instruction(qc)
qc.rx(2 * beta, 0)
final = Statevector.from_instruction(qc)
for name, state in [('초기', initial), ('비용 회로 후', after_cost), ('믹서 후', final)]:
    print(name, '진폭:', np.round(state.data, 6),
          '확률:', np.round(state.probabilities(), 6))
print('기대 비용:', float(final.probabilities() @ np.array([0.0, 1.0])))
assert np.allclose(final.probabilities(), [1, 0])

초기 진폭: [0.707107+0.j 0.707107+0.j] 확률: [0.5 0.5]
비용 회로 후 진폭: [0.707107+0.j       0.      -0.707107j] 확률: [0.5 0.5]
믹서 후 진폭: [1.+0.j 0.-0.j] 확률: [1. 0.]
기대 비용: 1.3263301508241975e-32


In [19]:
import time
from importlib.metadata import version

import numpy as np
from scipy.optimize import minimize
from threadpoolctl import threadpool_limits
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# 1. 실행 설정
threadpool_limits(limits=1)
started = time.perf_counter()
p = 2
maxiter = 100
shots = 4096
print('버전:', {k: version(k) for k in (
    'qiskit', 'qiskit-optimization', 'numpy', 'scipy', 'scikit-learn'
)})

# 2. 학습 데이터: [출석률%, 과제 수행률%, 시험점수, 주간 접속수, 중도포기]
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline

# 교육용 고정 사례입니다. 마지막 열은 0=완주, 1=중도포기입니다.
training = np.array([
    [95, 95, 90, 14, 0], [90, 85, 82, 12, 0],
    [85, 80, 78, 10, 0], [80, 76, 72,  9, 0],
    [75, 72, 68,  8, 0], [70, 66, 64,  7, 0],
    [88, 60, 73,  9, 0], [65, 82, 75,  8, 0],
    [92, 91, 85, 13, 0], [78, 85, 70, 10, 0],
    [55, 45, 40,  3, 1], [60, 50, 45,  4, 1],
    [62, 56, 52,  5, 1], [58, 48, 46,  4, 1],
    [67, 55, 58,  5, 1], [64, 58, 54,  5, 1],
    [72, 45, 48,  4, 1], [55, 70, 45,  4, 1],
    [48, 40, 35,  2, 1], [69, 59, 57,  6, 1],
], dtype=float)
testing = np.array([
    [93, 90, 86, 13, 0], [82, 78, 76, 10, 0],
    [74, 69, 66,  8, 0], [66, 80, 74,  9, 0],
    [57, 47, 43,  3, 1], [61, 53, 51,  5, 1],
    [71, 46, 49,  4, 1], [54, 68, 44,  4, 1],
], dtype=float)
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8), solver='lbfgs',
                  alpha=0.1, max_iter=500, random_state=7),
)
model.fit(training[:, :4], training[:, 4].astype(int))
pred = model.predict(testing[:, :4])
print('학습/시험 건수:', len(training), len(testing))
print('시험 정확도:', accuracy_score(testing[:, 4], pred))
print('혼동행렬 [실제 0,1 × 예측 0,1]:')
print(confusion_matrix(testing[:, 4], pred, labels=[0, 1]))
names = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8']
features = np.array([
    [92, 95, 88, 12], [75, 70, 65, 8],
    [60, 55, 50,  5], [85, 80, 78, 10],
    [58, 50, 48,  4], [68, 65, 62, 6],
    [95, 98, 91, 14], [72, 68, 60, 7],
], dtype=float)
risk = model.predict_proba(features)[:, 1]
assert np.isfinite(risk).all() and ((risk >= 0) & (risk <= 1)).all()
for name, value in zip(names, risk):
    print(name, '예측 위험도:', round(float(value), 6))
# 비용은 사전에 배정한 상담·튜터링 시간입니다.
cost = np.array([1, 2, 3, 2, 3, 2, 1, 2])
budget = 10
cost_unit = '시간'
single = False
cost_weight = 0.1
gain = 2.5 * risk
linear = cost_weight * cost - gain
pairs = {(2, 4): -1.5, (1, 5): -1.0, (3, 7): -0.8}


# 3. 목적함수와 업무 제약
m = len(names)
qp = QuadraticProgram()
for i in range(m):
    qp.binary_var(f'x{i}')
qp.minimize(linear=linear, quadratic=pairs)
qp.linear_constraint(
    {f'x{i}': int(cost[i]) for i in range(m)},
    '<=', int(budget), 'budget',
)
if single:
    qp.linear_constraint({f'x{i}': 1 for i in range(m)}, '==', 1, 'count')

# 4. QUBO와 Ising 변환
# 벌점은 원래 목적값의 가능한 변동 폭보다 크게 설정합니다.
penalty = float(np.abs(linear).sum() + sum(abs(v) for v in pairs.values()) + 1)
converter = QuadraticProgramToQubo(penalty=penalty)
qubo = converter.convert(qp)
H, offset = qubo.to_ising()
n = H.num_qubits
assert n <= 18, '큐비트가 18개를 초과했습니다. 비용 단위와 예산을 확인하세요.'
indices = np.arange(2**n)
bits = (indices[:, None] >> np.arange(n)) & 1
z = 1 - 2 * bits
terms = []
diagonal = np.zeros(2**n)
for label, coefficient in H.to_list():
    qubits = [i for i, char in enumerate(label[::-1]) if char == 'Z']
    assert set(label) <= {'I', 'Z'}
    coefficient = float(np.real(coefficient))
    terms.append((qubits, coefficient))
    diagonal += coefficient * np.prod(z[:, qubits], axis=1)
energy_values = diagonal + offset
qlinear = qubo.objective.linear.to_array()
qquad = qubo.objective.quadratic.to_dict()
qubo_values = qubo.objective.constant + bits @ qlinear
for (i, j), value in qquad.items():
    qubo_values += value * bits[:, i] * bits[:, j]
assert np.allclose(qubo_values, energy_values)
scale = max(1.0, float(np.max(np.abs(diagonal))))
print('변수 순서:', [v.name for v in qubo.variables])
print('큐비트:', n, '벌점:', round(penalty, 4))
print('QUBO–Ising 최대 오차:', float(np.max(abs(qubo_values - energy_values))))

# 5. 회로 구성: theta(세타)의 앞 p개는 감마, 뒤 p개는 베타입니다.
def circuit(theta):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        gamma = theta[layer]      # 감마: 정규화 비용의 위상 변화량
        beta = theta[p + layer]   # 베타: 믹서 회전량
        for qubits, coefficient in terms:
            angle = 2 * gamma * coefficient / scale
            if len(qubits) == 1:
                qc.rz(angle, qubits[0])
            elif len(qubits) == 2:
                qc.rzz(angle, *qubits)
        qc.rx(2 * beta, range(n))
    return qc

# 비용의 대각 연산과 RX를 배열로 계산하여 반복 평가 시간을 줄입니다.
def state(theta):
    psi = np.full(2**n, 1 / np.sqrt(2**n), dtype=complex)
    for layer in range(p):
        gamma, beta = theta[layer], theta[p + layer]
        psi *= np.exp(-1j * gamma * diagonal / scale)
        c, s = np.cos(beta), -1j * np.sin(beta)
        for q in range(n):
            view = psi.reshape(-1, 2, 2**q)
            left, right = view[:, 0].copy(), view[:, 1].copy()
            view[:, 0] = c * left + s * right
            view[:, 1] = s * left + c * right
    return psi

history = []
def energy(theta):
    probability = np.abs(state(theta))**2
    value = float(probability @ (diagonal / scale))
    history.append(value * scale + offset)
    return value

# 6. 고전 최적화: 고정 시작점 3개에서 기대 에너지를 최소화합니다.
results = []
for start in (0.2, 0.7, 1.2):
    initial = np.r_[np.full(p, start), np.full(p, 0.3)]
    result = minimize(
        energy, initial, method='COBYLA',
        options={'maxiter': maxiter, 'rhobeg': 0.4},
    )
    results.append(result)
best_run = min(results, key=lambda result: result.fun)
sv = Statevector.from_instruction(circuit(best_run.x))
fast_state = state(best_run.x)
fidelity = abs(np.vdot(sv.data, fast_state))**2
assert np.isclose(fidelity, 1.0, atol=1e-8)
probability = sv.probabilities()
assert np.isclose(probability.sum(), 1.0)
print('회로–배열 상태 충실도:', round(float(fidelity), 10))
print('목적함수 평가 수:', len(history), '종료 사유:', best_run.message)

# 7. 측정: 최종 회로에서 실제로 관측한 전체 비트열을 평가합니다.
sv.seed(7)
counts = sv.sample_counts(shots)
observed = {int(key, 2): int(value) for key, value in counts.items()}
chosen_index = min(observed, key=lambda k: (energy_values[k], k))
full_bits = bits[chosen_index]
x = np.rint(converter.interpret(full_bits)).astype(int)

def objective(v):
    return float(qp.objective.evaluate(v))

def feasible(v):
    return bool(qp.is_feasible(v))

# 8. 정확 최적값은 8개 업무 변수의 완전탐색으로 계산합니다.
# 측정 결과를 선택한 뒤 검증 기준을 계산합니다.
all_x = (np.arange(2**m)[:, None] >> np.arange(m)) & 1
valid_x = [v for v in all_x if feasible(v)]
exact = min(objective(v) for v in valid_x)
feasible_count = optimal_count = same_x_count = 0
for k, count in observed.items():
    v = np.rint(converter.interpret(bits[k])).astype(int)
    same_x_count += count * int(np.array_equal(v, x))
    if feasible(v):
        feasible_count += count
        if np.isclose(objective(v), exact):
            optimal_count += count

print('선택 비트:', x.tolist())
print('선택 항목:', [names[i] for i in range(m) if x[i]])
print('총 비용:', int(cost @ x), '/ 예산:', budget, cost_unit)
print('예산 제약 충족:', bool(cost @ x <= budget))
if single:
    print('정확히 한 명 선택:', bool(x.sum() == 1))
print('모든 업무 제약 충족:', feasible(x))
print('원래 목적함수:', round(objective(x), 6))
print('선택 QUBO 값:', round(float(energy_values[chosen_index]), 6))
print('기대 Ising 에너지:', round(float(probability @ diagonal), 6))
print('기대 QUBO 값:', round(float(probability @ energy_values), 6))
print('정확 최적값:', round(exact, 6))
print('최적값과 차이:', round(objective(x) - exact, 6) if feasible(x) else '제약 위반')
print('선택 전체 비트 관측 비율:', observed[chosen_index] / shots)
print('선택 업무 조합 관측 비율:', same_x_count / shots)
print('실행 가능한 해 관측 비율:', feasible_count / shots)
print('최적해 관측 비율:', optimal_count / shots)
print('실행 시간(초):', round(time.perf_counter() - started, 2))
assert sum(observed.values()) == shots
assert (not feasible(x)) or objective(x) >= exact - 1e-8

버전: {'qiskit': '2.2.3', 'qiskit-optimization': '0.7.0', 'numpy': '2.4.6', 'scipy': '1.16.3', 'scikit-learn': '1.6.1'}
학습/시험 건수: 20 8
시험 정확도: 1.0
혼동행렬 [실제 0,1 × 예측 0,1]:
[[4 0]
 [0 4]]
S1 예측 위험도: 0.0
S2 예측 위험도: 0.002885
S3 예측 위험도: 0.999707
S4 예측 위험도: 5e-06
S5 예측 위험도: 0.999982
S6 예측 위험도: 0.40027
S7 예측 위험도: 0.0
S8 예측 위험도: 0.098059
변수 순서: ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'budget@int_slack@0', 'budget@int_slack@1', 'budget@int_slack@2', 'budget@int_slack@3']
큐비트: 12 벌점: 10.1378
QUBO–Ising 최대 오차: 3.637978807091713e-12
회로–배열 상태 충실도: 1.0
목적함수 평가 수: 300 종료 사유: Return from COBYLA because the objective function has been evaluated MAXFUN times.
선택 비트: [0, 1, 1, 0, 1, 1, 0, 0]
선택 항목: ['S2', 'S3', 'S5', 'S6']
총 비용: 10 / 예산: 10 시간
예산 제약 충족: True
모든 업무 제약 충족: True
원래 목적함수: -7.507107
선택 QUBO 값: -7.507107
기대 Ising 에너지: -190.313701
기대 QUBO 값: 65.049548
정확 최적값: -7.507107
최적값과 차이: 0.0
선택 전체 비트 관측 비율: 0.000732421875
선택 업무 조합 관측 비율: 0.00390625
실행 가능한 해 관측 비율: 0.935546875
최적해 관측 비율: 0.00390625

이 실행에서는 S2·S3·S5·S6에 총 10시간을 배정합니다. 비용과 예산이 같고, 원래 목적값은 약 -7.507107입니다. 위험도 기반 보상 합에서 비용 점수와 쌍 보상을 반영한 값입니다. 시험 8건에서는 8건을 맞혔지만 데이터 규모와 구성이 제한되어 있습니다. 출력 위험도를 실제 현장의 중도포기 확률로 사용할 때는 별도 데이터 검증과 확률 보정이 필요합니다.

기대 Ising 에너지는 최종 분포 전체의 Ising 에너지 평균입니다. 기대 QUBO 값에는 상수 `offset`을 더합니다. 선택 QUBO 값은 관측된 특정 전체 비트열의 벌점 포함 비용입니다. 원래 목적함수는 업무 비트만으로 계산한 점수입니다. 예산 슬랙이 맞고 모든 제약을 충족하면 선택 QUBO 값과 원래 목적값이 일치합니다.

선택 전체 비트 관측 비율은 슬랙까지 동일한 비트열의 빈도입니다. 선택 업무 조합 관측 비율은 슬랙이 달라도 업무 선택이 같은 빈도를 합산한 값입니다. 실행 가능한 해 관측 비율과 최적해 관측 비율도 업무 변수 기준으로 집계합니다. 최적 업무 조합이 관측되어도 슬랙 값에 벌점이 붙으면 다른 전체 비트열이 선택될 수 있습니다. 이러한 경우에도 최종 선택을 그대로 평가합니다.

`Maximum number of function evaluations has been exceeded`는 정해진 목적함수 평가 한도에 도달했다는 종료 사유입니다. 해당 시점의 최선 각도로 측정과 결과 평가를 계속합니다. 이 출력만으로 최적화 수렴을 판정할 수 없습니다. 제약 충족, 최적값 차이와 관측 비율을 함께 확인합니다.

완전탐색은 8개 업무 변수의 256개 조합을 검사해 검증 기준을 구합니다. QAOA 선택 결과는 그 전에 최종 측정에서 결정됩니다. 최적해 관측 비율이 낮으면 측정 횟수나 각도에 따라 최종 선택이 달라질 수 있습니다.